# Notebook 4: Branch Risk Analysis

Objective: Identify branches likely to become loss-making.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

sns.set_style('whitegrid')

# Load branch KPIs
branch_kpis = pd.read_csv('../reports/branch_profitability_summary.csv')
branch_kpis['is_loss'] = (branch_kpis['net_profit'] < 0).astype(int)

# Features
features = ['total_revenue', 'profit_margin', 'cost_ratio', 'revenue_per_customer', 'volatility']  # assume volatility is std of profit
# For simplicity, add dummy volatility
branch_kpis['volatility'] = branch_kpis['profit_margin'] * 0.1  # placeholder

X = branch_kpis[features]
y = branch_kpis['is_loss']

# Train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Predict risk scores
branch_kpis['risk_score'] = model.predict_proba(X)[:, 1]  # prob of loss

# 12. Risk distribution
plt.figure(figsize=(10,6))
sns.histplot(branch_kpis['risk_score'], bins=10, kde=True)
plt.title('Risk Distribution')
plt.xlabel('Risk Score')
plt.savefig('../reports/12_risk_distribution.png')
plt.show()

In [ ]:
# High risk branches
high_risk = branch_kpis[branch_kpis['risk_score'] > 0.5][['branch_name', 'risk_score', 'main_issue']].rename(columns={'main_issue': 'key_issue'})
high_risk.to_csv('../reports/high_risk_branches.csv', index=False)
print('High risk branches saved.')

# Interpretation
print('Risk drivers: Low revenue and high costs. Branches like Mombasa need attention due to high risk from low demand.')